# iPLAID Pipeline

## Overview
This notebook runs the iPLAID pipeline directly from Python — the same logic that powers the web workbench. It reads `config/config.json`, the layout CSV, and either a metadata CSV or a self-contained source-plate-layout CSV, then writes dispense outputs for the configured dispenser (iDOT or Echo).

## What it does
1. **Loads configuration** from `config/config.json` (user, protocol, input file paths, dispenser, plate types).
2. **Reads layout + metadata**, or derives metadata from a source-plate-layout CSV when one is provided.
3. **Pre-flight validation** — checks every target concentration is feasible under the configured solvent caps.
4. **Stock selection + volume normalization** — applies solvent-family normalization while respecting per-solvent caps.
5. **Generates the dispenser-specific protocol CSV** (iDOT format or Echo format).
6. **Writes the liquids map and iMETA CSV.**
7. **Writes source-plate prep instructions** when no source layout is uploaded, or a **source-plate summary** when one is.

## Output files
- `iPLAID_{user}_{protocol}_{dispenser}_protocol_{ts}.csv` — dispenser-specific dispense protocol
- `iPLAID_{user}_{protocol}_liquids_map_{ts}.csv` — liquids map with source-plate well assignments
- `iPLAID_{user}_{protocol}_imeta_{ts}.csv` — iMETA metadata CSV
- `iPLAID_{user}_{protocol}_source_plate_prep_{ts}.txt` — source-plate prep instructions (no source layout uploaded), **or**
- `iPLAID_{user}_{protocol}_source_plate_summary_{ts}.txt` — source-plate summary (source layout uploaded)

## To run
Execute the cell below. All configuration is in `config/config.json`.

In [ ]:
from pathlib import Path

from iplaid.pipeline import run_pipeline

project_root = Path.cwd()
if not (project_root / "config" / "config.json").exists():
    project_root = project_root.parent

print("=" * 80)
print("iPLAID PIPELINE EXECUTION")
print("=" * 80)
print(f"Project root: {project_root}\n")

result = run_pipeline(project_root=project_root, include_source_prep=True)

print("\n" + "=" * 80)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 80)
print("\nResults summary:")
print(f"  - Dispenser:        {result['config']['dispenser']}")
print(f"  - Input compounds:  {len(result['df'])}")
print(f"  - Dispense rows:    {len(result['all_rows'])}")
print(f"  - Unique liquids:   {len(result['liquid_table_export'])}")
print(f"  - Solvent families: {len(result['solvent_summary'])}")
print(f"  - Max DMSO:         {result['max_dmso_ul']:.1f} uL")

print("\nOutput files:")
print(f"  - Protocol: {result['paths']['out_idot'].name}")
print(f"  - Liquids:  {result['paths']['out_liquids'].name}")
print(f"  - iMETA:    {result['paths']['out_imeta'].name}")
if result.get('source_layout_provided'):
    print("  - Source-plate summary written")
elif result.get('source_prep_volumes'):
    print("  - Source-plate prep instructions written")

print("\nPipeline execution successful.")
